# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [38]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [39]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
spotify_raw['Release Date'] = pd.to_datetime(
    spotify_raw['Release Date'], errors='coerce'
)

dim_tiempo = (
    spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
)

dim_tiempo['Year'] = dim_tiempo['Release Date'].dt.year
dim_tiempo['Month'] = dim_tiempo['Release Date'].dt.month
dim_tiempo['Day'] = dim_tiempo['Release Date'].dt.day

dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

dim_tiempo = dim_tiempo[['id_tiempo', 'Year', 'Month', 'Day', 'Release Date']]

print(f'Dimension Tiempo ({len(dim_tiempo)} fechas únicas):')
display(dim_tiempo.head())

Dimension Tiempo (1562 fechas únicas):


,id_tiempo,Year,Month,Day,Release Date
0,1,2024,4,26,2024-04-26
1,2,2024,5,4,2024-05-04
2,3,2024,3,19,2024-03-19
3,4,2023,1,12,2023-01-12
4,5,2024,5,31,2024-05-31


In [40]:
# 2. Crear dim_pista
dim_pista = (
    spotify_raw[['Track', 'Artist', 'Explicit Track']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_pista['id_pista'] = dim_pista.index + 1

dim_pista = dim_pista[['id_pista', 'Track', 'Artist', 'Explicit Track']]

In [41]:
# 3. Crear fact_streams
spotify_raw['Spotify Streams'] = pd.to_numeric(
    spotify_raw['Spotify Streams'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce',
)

spotify_raw['YouTube Views'] = pd.to_numeric(
    spotify_raw['YouTube Views'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce',
)

spotify_raw['Track Score'] = pd.to_numeric(
    spotify_raw['Track Score'], errors='coerce'
)

df_temp = spotify_raw.merge(
    dim_tiempo[['id_tiempo', 'Release Date']], on='Release Date', how='left'
).merge(
    dim_pista[['id_pista', 'Track', 'Artist', 'Explicit Track']],
    on=['Track', 'Artist', 'Explicit Track'],
    how='left',
)

metricas = ['Spotify Streams', 'YouTube Views', 'Track Score']
fact_streams = df_temp[['id_pista', 'id_tiempo'] + metricas].copy()

print(
    'Tabla de Hechos'
)
display(fact_streams.head())

Tabla de Hechos


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score
0,1,1,3.904709e+08,8.427475e+07,725.4
1,2,2,3.237039e+08,1.163470e+08,545.9
2,3,3,6.013093e+08,1.225991e+08,538.4
3,4,4,2.031281e+09,1.096101e+09,444.9
4,5,5,1.070349e+08,7.737396e+07,423.3


In [42]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_tiempo, on='id_tiempo', how='left').merge(
    dim_pista, on='id_pista', how='left'
)

### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

Solo hay 2 dimensiones por que simplificamos los atributos cancion, artista y cancion grosera, agrupando las jerarquias, podemos separar el artista en su propia dimension para saber cosas de ese cantante sin repetirlo varias veces en sus varios hits, lo mismo para el album y la plataforma

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [43]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll up
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum()
print('RETO 1')
display(respuesta_1.sort_values(ascending=False).head())

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice
respuesta_2 = cubo_olap[cubo_olap['Year'] == 2024]
print('RETO 2')
display(respuesta_2.head(3))

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP:  Drill Down
respuesta_3 = cubo_olap.groupby(['Year', 'Month'])['Spotify Streams'].sum()
print('RETO 3')
display(respuesta_3.tail(6))

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice
respuesta_4 = cubo_olap[
    (cubo_olap['Explicit Track'] == 1) & (cubo_olap['Year'] == 2024)
]
print('RETO 4')
display(respuesta_4.head(3))

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll up
respuesta_5 = cubo_olap.groupby('Year')['Track Score'].mean()
print('RETO 5')
display(respuesta_5)

RETO 1


Artist
Bad Bunny       3.705483e+10
The Weeknd      3.694854e+10
Drake           3.496216e+10
Taylor Swift    3.447077e+10
Post Malone     2.613747e+10
Name: Spotify Streams, dtype: float64

RETO 2


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Year,Month,Day,Release Date,Track,Artist,Explicit Track
0,1,1,390470936.0,84274754.0,725.4,2024,4,26,2024-04-26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,323703884.0,116347040.0,545.9,2024,5,4,2024-05-04,Not Like Us,Kendrick Lamar,1
2,3,3,601309283.0,122599116.0,538.4,2024,3,19,2024-03-19,i like the way you kiss me,Artemas,0


RETO 3


Year  Month
2024  1        1.224484e+10
      2        1.023077e+10
      3        1.263673e+10
      4        2.152035e+10
      5        1.005348e+10
      6        2.711085e+09
Name: Spotify Streams, dtype: float64

RETO 4


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Year,Month,Day,Release Date,Track,Artist,Explicit Track
1,2,2,323703884.0,116347040.0,545.9,2024,5,4,2024-05-04,Not Like Us,Kendrick Lamar,1
4,5,5,107034922.0,77373957.0,423.3,2024,5,31,2024-05-31,Houdini,Eminem,1
7,8,8,675079153.0,228382568.0,375.8,2024,2,2,2024-02-02,Gata Only,FloyyMenor,1


RETO 5


Year
1987    21.500000
1991    19.600000
1994    31.500000
1998    20.800000
1999    22.200000
2000    30.400000
2001    20.100000
2002    26.060000
2003    23.766667
2004    24.240000
2005    27.140000
2006    22.300000
2007    23.860000
2008    23.854545
2009    26.000000
2010    31.425926
2011    31.800000
2012    32.197872
2013    31.674545
2014    31.678571
2015    37.988542
2016    37.828455
2017    37.949765
2018    36.977510
2019    39.380405
2020    38.042222
2021    38.144118
2022    41.693228
2023    43.264594
2024    54.120058
Name: Track Score, dtype: float64

In [44]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
print('RETO 6')
display(respuesta_6.head(3))

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill Down
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum()
print('RETO 7')
display(respuesta_7.sort_values(ascending=False).head(6))

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice
respuesta_8 = cubo_olap[(cubo_olap['Month'] == 12) & (cubo_olap['Year'] == 2023)]
print('RETO 8')
display(respuesta_8.head(3))

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll up
respuesta_9 = cubo_olap.groupby('Month')['Spotify Streams'].sum()
print('RETO 9')
display(respuesta_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice  
respuesta_10 = cubo_olap[
    (cubo_olap['Artist'] == 'Taylor Swift') &
    (cubo_olap['Explicit Track'] == 0) &
    (cubo_olap['Year'] == 2024)
]
print('RETO 10')
display(respuesta_10.head(3))

RETO 6


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Year,Month,Day,Release Date,Track,Artist,Explicit Track
63,61,57,1.811991e+09,1.516863e+09,184.8,2022,5,6,2022-05-06,Me Porto Bonito,Bad Bunny,1
88,86,57,1.514240e+09,1.366181e+09,163.1,2022,5,6,2022-05-06,Titï¿½ï¿½ Me Pregu,Bad Bunny,1
102,100,91,4.781509e+08,1.133797e+08,150.7,2023,10,13,2023-10-13,MONACO,Bad Bunny,1


RETO 7


Artist          Track                             
Pinkfong        Baby Shark                            1.632276e+10
Luis Fonsi      Despacito                             9.537136e+09
Ed Sheeran      Shape of You                          8.013997e+09
Kids With Buns  bathroom floor                        6.736502e+09
Wiz Khalifa     See You Again (feat. Charlie Puth)    6.584796e+09
Ed Sheeran      Perfect                               6.473699e+09
Name: YouTube Views, dtype: float64

RETO 8


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Year,Month,Day,Release Date,Track,Artist,Explicit Track
61,59,55,1384.0,222076.0,186.5,2023,12,31,2023-12-31,Very Cute Melody by Marimba Tone (39813),mitsu sound,0
94,92,84,649404552.0,433839212.0,155.9,2023,12,2,2023-12-02,La Diabla,Xavi,0
97,95,87,359803580.0,369321334.0,153.1,2023,12,7,2023-12-07,BELLAKEO,Peso Pluma,1


RETO 9


Month
1     3.652168e+11
2     1.333565e+11
3     1.638041e+11
4     1.462846e+11
5     1.779293e+11
6     1.484293e+11
7     1.503832e+11
8     1.426826e+11
9     1.363626e+11
10    1.672667e+11
11    1.734614e+11
12    1.022498e+11
Name: Spotify Streams, dtype: float64

RETO 10


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Year,Month,Day,Release Date,Track,Artist,Explicit Track
16,17,17,395433400.0,104673836.0,297.6,2024,4,18,2024-04-18,Fortnight (feat. Post Malone),Taylor Swift,0
866,861,17,166168092.0,16423009.0,52.3,2024,4,18,2024-04-18,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0
1013,1007,17,175613480.0,15237337.0,47.8,2024,4,18,2024-04-18,"So Long, London",Taylor Swift,0
